**Set environment**

In [1]:
suppressMessages(suppressWarnings(source("../run_config_project.R")))
suppressMessages(suppressWarnings(library("xgboost")))
suppressMessages(suppressWarnings(library("Matrix")))
show_env()

BASE DIRECTORY (FD_BASE): /hpc/group/igvf/kk319 
REPO DIRECTORY (FD_REPO): /hpc/group/igvf/kk319/repo 
WORK DIRECTORY (FD_WORK): /hpc/group/igvf/kk319/work 
DATA DIRECTORY (FD_DATA): /hpc/group/igvf/kk319/data 

You are working with      IGVF BlueSTARR 
PATH OF PROJECT (FD_PRJ): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR 
PROJECT RESULTS (FD_RES): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results 
PROJECT SCRIPTS (FD_EXE): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts 
PROJECT DATA    (FD_DAT): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data 
PROJECT NOTE    (FD_NBK): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks 
PROJECT DOCS    (FD_DOC): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs 
PROJECT LOG     (FD_LOG): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log 
PROJECT REF     (FD_REF): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references 



## Import data

**Check file existence**

In [2]:
### set file directory
txt_folder = "motifmodel_pilot_jvierstra_v2.1beta"
txt_fdiry  = file.path(FD_RES, "analysis_variant_motif_richard", txt_folder)

vec = dir(txt_fdiry)
for (txt in vec){cat(txt, "\n")}

variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot.nuc.X.rds 
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot.ori.X.rds 
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot.xgb_cv.rds 
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot.xgb_model.bin 
variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.pilot.y.rds 


### Import data

In [3]:
### set file directory
txt_prefix = "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
txt_folder = "motifmodel_pilot_jvierstra_v2.1beta"
txt_fdiry  = file.path(FD_RES, "analysis_variant_motif_richard", txt_folder)
txt_fname  = paste(txt_prefix, "pilot.y.rds", sep = ".")
txt_fpath  = file.path(txt_fdiry, txt_fname)

### read table
obj = read_rds(txt_fpath)

### assign and show
vec_y_ori_import = obj
print(length(obj))
print(head(obj))

[1] 199971
chr1:100186033:T:T:A chr1:100306767:A:A:C chr1:100307737:G:G:C 
           0.1147592            0.1031407            0.1006881 
chr1:100324154:G:G:C chr1:100520609:T:C:G chr1:100521850:A:A:T 
           0.0812454            0.1144301            0.1018046 


In [4]:
### set file directory
txt_prefix = "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
txt_folder = "motifmodel_pilot_jvierstra_v2.1beta"
txt_fdiry  = file.path(FD_RES, "analysis_variant_motif_richard", txt_folder)
txt_fname  = paste(txt_prefix, "pilot.ori.X.rds", sep = ".")
txt_fpath  = file.path(txt_fdiry, txt_fname)

### read table
dat = read_rds(txt_fpath)

### assign and show
mat_event_ori_import = dat
print(dim(dat))
fun_display_table(head(dat[,1:6], 3))

[1] 199971    629


,AC0001:GATA/PROP:GATA,AC0002:PROP/ALX:Homeodomain,AC0003:HNF1A/HNF1B:Homeodomain,AC0004:ZSCAN:C2H2_ZF,"AC0005:POU3F/POU1F:Homeodomain,POU",AC0006:MEOX:Homeodomain
chr1:100186033:T:T:A,0,0,0,0,0,0
chr1:100306767:A:A:C,0,0,0,0,0,0
chr1:100307737:G:G:C,0,0,0,0,0,0


In [5]:
### set file directory
txt_prefix = "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
txt_folder = "motifmodel_pilot_jvierstra_v2.1beta"
txt_fdiry  = file.path(FD_RES, "analysis_variant_motif_richard", txt_folder)
txt_fname  = paste(txt_prefix, "pilot.nuc.X.rds", sep = ".")
txt_fpath  = file.path(txt_fdiry, txt_fname)

### read table
dat = read_rds(txt_fpath)

### assign and show
mat_event_nuc_import = dat
print(dim(dat))
fun_display_table(head(dat[,1:6], 3))

[1] 199971    629


,AC0001:GATA/PROP:GATA,AC0002:PROP/ALX:Homeodomain,AC0003:HNF1A/HNF1B:Homeodomain,AC0004:ZSCAN:C2H2_ZF,"AC0005:POU3F/POU1F:Homeodomain,POU",AC0006:MEOX:Homeodomain
chr1:100186033:T:T:A,0,0,0,0,0,0
chr1:100306767:A:A:C,0,0,0,0,0,0
chr1:100307737:G:G:C,0,0,0,0,0,0


## Prepare

**Convert X to sparse**

In [6]:
### convert to sparse dgCMatrix
mat_X_ori = mat_event_ori_import
mat_X_nuc = mat_event_nuc_import
vec_y     = vec_y_ori_import

mat_X_sparse_ori = Matrix::Matrix(mat_X_ori, sparse = TRUE)
mat_X_sparse_nuc = Matrix::Matrix(mat_X_nuc, sparse = TRUE)

### check and remove matrix to save memory
stopifnot(nrow(mat_X_sparse_ori) == length(vec_y))
stopifnot(nrow(mat_X_sparse_nuc) == length(vec_y))
rm(mat_event_ori_import)
rm(mat_event_nuc_import)
gc()

### show results
mat = mat_X_sparse_ori
print(class(mat))
print(dim(mat))
mat = mat_X_sparse_nuc
print(class(mat))
print(dim(mat))

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2539121,135.7,4218671,225.4,4218671,225.4
Vcells,269839806,2058.8,372758007,2844.0,271711366,2073.0


[1] "dgCMatrix"
attr(,"package")
[1] "Matrix"
[1] 199971    629
[1] "dgCMatrix"
attr(,"package")
[1] "Matrix"
[1] 199971    629


**Build DMatrix**

In [7]:
dtrain_ori = xgboost::xgb.DMatrix(
    data  = mat_X_sparse_ori,
    label = vec_y
)

dtrain_nuc = xgboost::xgb.DMatrix(
    data  = mat_X_sparse_nuc,
    label = vec_y
)

## Cross-validation

**Define parameters**

In [ ]:
param = list(
    objective        = "reg:squarederror",
    eval_metric      = "rmse",
    max_depth        = 6,
    eta              = 0.1,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    nthread          = 8
)

**Original sequence**

In [7]:
### start timer
time_start = Sys.time()

### init seed
set.seed(123)

fit_xgb_ori_cv = xgboost::xgb.cv(
    params = param,
    data   = dtrain_ori,
    nrounds = 3000,
    nfold   = 5,
    early_stopping_rounds = 50,
    verbose = 0
)

### end timer
time_end = Sys.time()

### print elapsed time
print(time_end - time_start)

Time difference of 26.44866 mins


**Dinuc shuffle sequence**

In [ ]:
### start timer
time_start = Sys.time()

### init seed
set.seed(123)

fit_xgb_nuc_cv = xgboost::xgb.cv(
    params = param,
    data   = dtrain_nuc,
    nrounds = 3000,
    nfold   = 5,
    early_stopping_rounds = 50,
    verbose = 0
)

### end timer
time_end = Sys.time()

### print elapsed time
print(time_end - time_start)

In [8]:
### assign and show
best_iter = fit_xgb_cv$best_iteration
best_test_rmse = fit_xgb_cv$evaluation_log$test_rmse_mean[best_iter]
print(best_iter)
print(best_test_rmse)

vec = fit_xgb_cv$evaluation_log
print(head(vec, 100))
cat("\n")
print(tail(vec, 100))

[1] 2568
[1] 0.05646075
      iter train_rmse_mean train_rmse_std test_rmse_mean test_rmse_std
     <num>           <num>          <num>          <num>         <num>
  1:     1      0.40139255   4.600811e-05     0.40141733  0.0002136557
  2:     2      0.36238194   3.396629e-05     0.36243152  0.0002129574
  3:     3      0.32739716   3.198234e-05     0.32748243  0.0001985589
  4:     4      0.29601790   2.804590e-05     0.29613950  0.0002136397
  5:     5      0.26790404   2.042242e-05     0.26807141  0.0002056657
  6:     6      0.24272160   1.188340e-05     0.24293636  0.0002065968
  7:     7      0.22020287   1.292459e-05     0.22047750  0.0002106083
  8:     8      0.20009010   3.123287e-05     0.20042849  0.0002186316
  9:     9      0.18217461   4.262035e-05     0.18258122  0.0002357375
 10:    10      0.16621529   4.701223e-05     0.16670456  0.0002458223
 11:    11      0.15205252   5.752425e-05     0.15261470  0.0002670482
 12:    12      0.13950129   7.499411e-05     0.14015

## Build final model

In [9]:
### start timer
time_start = Sys.time()

best_iter = fit_xgb_cv$best_iteration
fit_xgb_final = xgboost::xgb.train(
    params  = param,
    data    = dtrain,
    nrounds = best_iter,
    verbose = 1
)

### end timer and show elapsed time
time_end = Sys.time()
print(time_end - time_start)

Time difference of 5.98458 mins


In [10]:
### --------------------------------
### save CV result
### --------------------------------
txt_prefix = "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
txt_folder = "motifmodel_pilot_jvierstra_v2.1beta"
txt_fdiry  = file.path(FD_RES, "analysis_variant_motif_richard", txt_folder)

txt_fname  = paste(txt_prefix, "pilot.xgb_cv.rds", sep=".")
txt_fpath  = file.path(txt_fdiry, txt_fname)

readr::write_rds(fit_xgb_cv, txt_fpath)

In [11]:
### --------------------------------
### save final model
### --------------------------------
txt_fname  = paste(txt_prefix, "pilot.xgb_model.bin", sep=".")
txt_fpath  = file.path(txt_fdiry, txt_fname)

xgboost::xgb.save(fit_xgb_final, txt_fpath)

[1] TRUE

Later import
```
fit_xgb_cv = readr::read_rds(txt_fpath)
fit_xgb_final = xgboost::xgb.load(txt_fpath)
```

**Define Parameters**

In [11]:
param = list(
    objective        = "reg:squarederror",
    eval_metric      = "rmse",
    max_depth        = 6,
    eta              = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    nthread          = 8
)

## Cross-validation

In [12]:
### start timer
time_start = Sys.time()

### init seed
set.seed(123)

fit_xgb_cv = xgboost::xgb.cv(
    params = param,
    data   = dtrain,
    nrounds = 2000,
    nfold   = 5,
    early_stopping_rounds = 50,
    verbose = 1
)

### assign and show
best_iter = fit_xgb_cv$best_iteration
print(best_iter)
print(fit_xgb_cv$best_score)

### end timer
time_end = Sys.time()

### print elapsed time
print(time_end - time_start)

[1]	train-rmse:0.423112+0.000049	test-rmse:0.423124+0.000215 
Multiple eval metrics are present. Will use test_rmse for early stopping.
Will train until test_rmse hasn't improved in 50 rounds.

[2]	train-rmse:0.402482+0.000042	test-rmse:0.402504+0.000215 
[3]	train-rmse:0.382919+0.000039	test-rmse:0.382956+0.000212 
[4]	train-rmse:0.364353+0.000027	test-rmse:0.364403+0.000218 
[5]	train-rmse:0.346744+0.000025	test-rmse:0.346810+0.000213 
[6]	train-rmse:0.330032+0.000022	test-rmse:0.330114+0.000210 
[7]	train-rmse:0.314183+0.000022	test-rmse:0.314285+0.000202 
[8]	train-rmse:0.299151+0.000015	test-rmse:0.299274+0.000208 
[9]	train-rmse:0.284910+0.000017	test-rmse:0.285054+0.000214 
[10]	train-rmse:0.271405+0.000020	test-rmse:0.271572+0.000209 
[11]	train-rmse:0.258606+0.000022	test-rmse:0.258792+0.000207 
[12]	train-rmse:0.246483+0.000022	test-rmse:0.246691+0.000212 
[13]	train-rmse:0.235008+0.000016	test-rmse:0.235239+0.000204 
[14]	train-rmse:0.224137+0.000016	test-rmse:0.224393+0.000

In [14]:
param = list(
    objective        = "reg:squarederror",
    eval_metric      = "rmse",
    max_depth        = 6,
    eta              = 0.1,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    nthread          = 8
)

### start timer
time_start = Sys.time()

### init seed
set.seed(123)

fit_xgb_cv = xgboost::xgb.cv(
    params = param,
    data   = dtrain,
    nrounds = 1000,
    nfold   = 5,
    early_stopping_rounds = 50,
    verbose = 1
)

### assign and show
best_iter = fit_xgb_cv$best_iteration
print(best_iter)
print(fit_xgb_cv$best_score)

### end timer
time_end = Sys.time()

### print elapsed time
print(time_end - time_start)

[1]	train-rmse:0.401393+0.000046	test-rmse:0.401417+0.000214 
Multiple eval metrics are present. Will use test_rmse for early stopping.
Will train until test_rmse hasn't improved in 50 rounds.

[2]	train-rmse:0.362382+0.000034	test-rmse:0.362432+0.000213 
[3]	train-rmse:0.327397+0.000032	test-rmse:0.327482+0.000199 
[4]	train-rmse:0.296018+0.000028	test-rmse:0.296140+0.000214 
[5]	train-rmse:0.267904+0.000020	test-rmse:0.268071+0.000206 
[6]	train-rmse:0.242722+0.000012	test-rmse:0.242936+0.000207 
[7]	train-rmse:0.220203+0.000013	test-rmse:0.220478+0.000211 
[8]	train-rmse:0.200090+0.000031	test-rmse:0.200428+0.000219 
[9]	train-rmse:0.182175+0.000043	test-rmse:0.182581+0.000236 
[10]	train-rmse:0.166215+0.000047	test-rmse:0.166705+0.000246 
[11]	train-rmse:0.152053+0.000058	test-rmse:0.152615+0.000267 
[12]	train-rmse:0.139501+0.000075	test-rmse:0.140153+0.000279 
[13]	train-rmse:0.128429+0.000069	test-rmse:0.129175+0.000307 
[14]	train-rmse:0.118663+0.000082	test-rmse:0.119513+0.000

In [16]:
print(fit_xgb_cv$niter)
print(fit_xgb_cv$best_iteration)
print(fit_xgb_cv$best_score)
tail(fit_xgb_cv$evaluation_log, 3)

[1] 1000
[1] 999
NULL


iter,train_rmse_mean,train_rmse_std,test_rmse_mean,test_rmse_std
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
998,0.04738894,0.0006402942,0.05676356,0.0007030246
999,0.04738169,0.0006440357,0.05676322,0.0007028895
1000,0.04738169,0.0006440368,0.05676324,0.0007028473


## Train final model

In [13]:
best_iter

[1] 1997

In [ ]:
### start timer
time_start = Sys.time()

fit_xgb = xgboost::xgb.train(
    params  = param,
    data    = dtrain,
    nrounds = best_iter,
    verbose = 1
)

### end timer and show elapsed time
time_end = Sys.time()
print(time_end - time_start)

## Feature Importance

In [ ]:
dat_imp = xgboost::xgb.importance(model = fit_xgb)

head(dat_imp, 20)